# SmartHome Care Copilot — 데모 (`demo.ipynb`)

제품 탑재형 고객지원/원격진단 에이전트: **Router → Structured Output(JSON) → Tool loop → RAG(출처 태그) → Handoff**.

- 위험 작업(펌웨어/공장초기화): **2회 사용자 확인** 후에만 실행
- **역할(customer / support / admin)** 에 따른 툴 가드
- 매뉴얼 **NO_HIT** 시 모델·증상·에러코드 등 추가 질문

## 가상환경 ( `project1_agent/.venv` )

```powershell
cd project1_agent
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r ..\requirements.txt
```

노트북 커널도 위 `.venv` 의 Python 으로 선택하세요.

## 의존성 (선택: 커널에 패키지가 없을 때만 1회 실행)

`project1_agent` 를 작업 디렉터리로 두고 실행하는 것을 권장합니다.

이 데모는 **pandas**, **pydantic**만 사용합니다. `requirements.txt` 전체가 다른 패키지와 충돌하면 위 두 패키지를 설치해도 됩니다. **Python 3.10 이상**을 권장합니다.

In [1]:
%pip install -r ../requirements.txt

INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached openai-2.24.0-py3-none-any.whl.metadata (29 kB)
  Using cached langchain_core-1.2.18-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain-1.2.12-py3-none-any.whl.metadata (5.7 kB)
  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install langchain-openai==1.1.11 and openai==2.24.0 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested openai==2.24.0
    langchain-opena

In [2]:
import json
import re
import uuid
from pathlib import Path
from typing import Any, Literal, Optional

import pandas as pd
from pydantic import BaseModel, Field

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MANUAL_DIR = DATA_DIR / "manuals"


def load_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)


devices: dict[str, Any] = load_json(DATA_DIR / "devices.json")
events: dict[str, list[dict[str, Any]]] = load_json(DATA_DIR / "events.json")
policies = load_json(DATA_DIR / "policies.json")
scenarios = load_json(DATA_DIR / "scenarios.json")
tickets: list[dict[str, Any]] = load_json(DATA_DIR / "tickets.json")

pd.DataFrame(scenarios)

,id,input,expected_intent
0,scenario-1,세탁기 5C 에러가 떠요. 원격진단하고 조치 알려줘,troubleshooting
1,scenario-2,로봇청소기 흡입이 약해요. 필터 점검 순서 알려줘,troubleshooting
2,scenario-3,앱 연결이 안 돼요. 와이파이 다시 붙이는 방법 알려줘,manual_info
3,scenario-4,에어컨 펌웨어 업데이트 해줘,device_control
4,scenario-5,탄 냄새랑 과열이 느껴져요,handoff


## Step 1. Structured Output 스키마 (`DeviceActionPlan`, `SupportReport`)

In [3]:
Role = Literal["customer", "support", "admin"]
Intent = Literal["manual_info", "device_control", "troubleshooting", "handoff"]


class DeviceActionPlan(BaseModel):
    intent: Intent
    device_id: Optional[str] = None
    device_type: Optional[str] = None
    action: Optional[str] = None
    question: Optional[str] = None
    symptom: Optional[str] = None
    error_code: Optional[str] = None
    safety_risk: bool = False
    confirmed: bool = False
    missing_fields: list[str] = Field(default_factory=list)


class SupportReport(BaseModel):
    title: str
    intent: Intent
    severity: Literal["low", "medium", "high"]
    reply: str
    evidence: list[str] = Field(default_factory=list)
    recommended_actions: list[str] = Field(default_factory=list)
    follow_up_question: Optional[str] = None
    needs_handoff: bool = False
    needs_confirmation: bool = False
    pending_action: Optional[dict[str, Any]] = None
    ticket: Optional[dict[str, Any]] = None
    tool_trace: list[dict[str, Any]] = Field(default_factory=list)


SupportReport(
    title="schema_check",
    intent="manual_info",
    severity="low",
    reply="ok",
)

SupportReport(title='schema_check', intent='manual_info', severity='low', reply='ok', evidence=[], recommended_actions=[], follow_up_question=None, needs_handoff=False, needs_confirmation=False, pending_action=None, ticket=None, tool_trace=[])

## Step 2. 라우팅 헬퍼 (키워드·디바이스 추출)

In [4]:
DEVICE_KEYWORDS = {
    "robot_vacuum": ["로봇청소기", "청소기", "vacuum"],
    "washing_machine": ["세탁기", "washer"],
    "air_conditioner": ["에어컨", "aircon", "ac"],
    "remote_hub": ["앱", "와이파이", "wifi", "wi-fi", "허브", "연결"],
}

SAFETY_KEYWORDS = [
    "연기",
    "과열",
    "탄 냄새",
    "타는 냄새",
    "가스",
    "누전",
    "불꽃",
    "스파크",
]
YES_WORDS = ["예", "네", "응", "확인", "진행", "동의", "yes", "y", "ok"]
NO_WORDS = ["아니", "아니오", "취소", "중단", "no", "n"]

DEVICE_ID_RE = re.compile(r"DEV-\d{4}")

MANUAL_BY_DEVICE_TYPE: dict[str, list[str]] = {
    "washing_machine": ["manual_washer.txt"],
    "robot_vacuum": ["manual_vacuum.txt"],
    "remote_hub": ["manual_remote.txt"],
    "air_conditioner": ["manual_air_conditioner.txt"],
}


def detect_device_type(message: str) -> Optional[str]:
    lowered = message.lower()
    for device_type, keywords in DEVICE_KEYWORDS.items():
        if any(keyword in lowered for keyword in keywords):
            return device_type
    return None


def resolve_device_id(device_type: Optional[str]) -> Optional[str]:
    if device_type is None:
        return None
    for device_id, device in devices.items():
        if device.get("device_type") == device_type:
            return device_id
    return None


def _tokenize_for_overlap(text: str) -> list[str]:
    return re.findall(r"[\w가-힣]+", text.lower())


def _is_yes(message: str) -> bool:
    t = message.strip().lower()
    return any(w in message for w in YES_WORDS) or t in {"y", "ok"}


def _is_no(message: str) -> bool:
    return any(w in message for w in NO_WORDS)


def role_may_use_sensitive_tools(role: Role) -> bool:
    return role in ("support", "admin")


def extract_error_code(message: str) -> Optional[str]:
    if re.search(r"\b5c\b", message, re.I):
        return "5C"
    m = re.search(r"\b([eE]\d{1,3})\b", message)
    if m:
        return m.group(1).upper()
    return None

## Step 3. Manual RAG — `search_manual(query, device_type)` (출처 citation 강제)

In [5]:
_MANUAL_CHUNKS: list[dict[str, str]] | None = None


def _load_manual_chunks() -> list[dict[str, str]]:
    global _MANUAL_CHUNKS
    if _MANUAL_CHUNKS is not None:
        return _MANUAL_CHUNKS
    chunks: list[dict[str, str]] = []
    for path in sorted(MANUAL_DIR.glob("manual_*.txt")):
        stem = path.stem
        text = path.read_text(encoding="utf-8")
        parts = re.split(r"\n\s*\n", text.strip())
        idx = 0
        for part in parts:
            body = " ".join(part.splitlines()).strip()
            if not body:
                continue
            idx += 1
            citation = f"[manual:{stem}:chunk{idx}]"
            chunks.append({"citation": citation, "manual_stem": stem, "text": body})
    _MANUAL_CHUNKS = chunks
    return chunks


def search_manual(query: str, device_type: Optional[str], top_k: int = 3) -> list[dict[str, str]]:
    """키워드 겹침 점수 기반 더미 RAG. 반환: [{"citation", "text"}, ...]"""
    all_chunks = _load_manual_chunks()
    preferred = set(MANUAL_BY_DEVICE_TYPE.get(device_type or "", []))
    q_tokens = set(_tokenize_for_overlap(query))
    if not q_tokens:
        q_tokens = set(query.lower())

    def score_row(ch: dict[str, str]) -> float:
        t = ch["text"].lower()
        s = sum(1 for tok in q_tokens if len(tok) > 1 and tok in t)
        s += 0.3 * sum(1 for tok in q_tokens if len(tok) == 1 and tok in t)
        fname = ch["manual_stem"] + ".txt"
        if preferred and fname in preferred:
            s += 2.0
        return s

    ranked = sorted(all_chunks, key=score_row, reverse=True)
    best = score_row(ranked[0]) if ranked else 0.0
    if best <= 0:
        return []
    out: list[dict[str, str]] = []
    for ch in ranked:
        if score_row(ch) <= 0:
            continue
        out.append({"citation": ch["citation"], "text": ch["text"]})
        if len(out) >= top_k:
            break
    return out

## Step 4. 더미 Tool (상태·이벤트·진단·설정·티켓·펌웨어·공장초기화)

In [6]:
def get_device_status(device_id: str) -> dict[str, Any]:
    if device_id not in devices:
        return {"ok": False, "error": {"code": "DEVICE_NOT_FOUND", "message": device_id}}
    return {"ok": True, "device_id": device_id, "status": devices[device_id]}


def get_recent_events(device_id: str, limit: int = 3) -> dict[str, Any]:
    if device_id not in events:
        return {"ok": False, "error": {"code": "EVENTS_NOT_FOUND", "message": device_id}}
    ev = events[device_id][-limit:]
    return {"ok": True, "device_id": device_id, "events": ev}


def run_remote_diagnosis(device_id: str) -> dict[str, Any]:
    st = get_device_status(device_id)
    if not st["ok"]:
        return st
    d = st["status"]
    dtype = d.get("device_type")
    findings: list[str] = []
    if dtype == "washing_machine":
        has_5c = d.get("last_error") == "5C" or any(
            "5C" in str(e.get("message", "")) for e in events.get(device_id, [])[-5:]
        )
        if has_5c:
            findings.append("배수 타임아웃(5C) 패턴: 드레인 라인/필터 막힘 가능성이 높습니다.")
    if dtype == "robot_vacuum" and d.get("suction_health") == "low":
        findings.append("흡입 저하: 필터 적재 시간 초과 및 브러시 먼지 의심")
    if dtype == "remote_hub" and d.get("wifi_status") == "disconnected":
        findings.append("Wi-Fi 미연결: SSID/밴드(2.4GHz) 및 공유기 상태를 점검하세요.")
    if dtype == "air_conditioner" and d.get("filter_warning"):
        findings.append("필터 경고 활성: 필터 세척/건조 후 재장착을 권장합니다.")
    if not findings:
        findings.append("특이 징후 없음: 최근 이벤트와 전원/연결 상태를 추가 확인하세요.")
    return {"ok": True, "device_id": device_id, "findings": findings, "raw": d}


def set_device_setting(device_id: str, setting: str, value: Any) -> dict[str, Any]:
    if device_id not in devices:
        return {"ok": False, "error": {"code": "DEVICE_NOT_FOUND", "message": device_id}}
    devices[device_id][setting] = value
    return {"ok": True, "device_id": device_id, "applied": {setting: value}}


def create_service_ticket(device_id: str, summary: str, severity: str) -> dict[str, Any]:
    tid = f"TKT-{uuid.uuid4().hex[:8].upper()}"
    ticket = {
        "id": tid,
        "device_id": device_id,
        "summary": summary,
        "severity": severity,
    }
    tickets.append(ticket)
    return {"ok": True, "ticket": ticket}


def update_firmware(device_id: str, target_version: str, confirmed: bool = False) -> dict[str, Any]:
    if device_id not in devices:
        return {"ok": False, "error": {"code": "DEVICE_NOT_FOUND", "message": device_id}}
    if not confirmed:
        return {
            "ok": False,
            "error": {"code": "confirmation_required", "message": "펌웨어 업데이트는 사용자 2회 확인 후 실행됩니다."},
        }
    devices[device_id]["fw"] = target_version
    return {"ok": True, "device_id": device_id, "fw": target_version}


def factory_reset(device_id: str, confirmed: bool = False) -> dict[str, Any]:
    if device_id not in devices:
        return {"ok": False, "error": {"code": "DEVICE_NOT_FOUND", "message": device_id}}
    if not confirmed:
        return {
            "ok": False,
            "error": {"code": "confirmation_required", "message": "공장 초기화는 사용자 2회 확인 후 실행됩니다."},
        }
    return {"ok": True, "device_id": device_id, "reset": "completed_simulation"}

## Step 5. Router — `build_action_plan(message)` → `DeviceActionPlan`

In [7]:
def build_action_plan(message: str) -> DeviceActionPlan:
    msg = message.strip()
    lowered = msg.lower()

    device_id_m = DEVICE_ID_RE.search(msg)
    device_id = device_id_m.group(0) if device_id_m else None
    device_type = detect_device_type(msg)
    if device_id is None and device_type:
        device_id = resolve_device_id(device_type)

    error_code = extract_error_code(msg)
    missing: list[str] = []

    if any(k in msg for k in SAFETY_KEYWORDS):
        if device_type is None:
            missing.append("device_type_or_product")
        return DeviceActionPlan(
            intent="handoff",
            device_id=device_id,
            device_type=device_type,
            symptom=msg,
            safety_risk=True,
            missing_fields=missing,
        )

    if re.search(r"펌웨어|firmware", lowered) and re.search(r"업데이트|update", lowered):
        if device_type is None:
            missing.append("device_type")
        return DeviceActionPlan(
            intent="device_control",
            device_id=device_id,
            device_type=device_type,
            action="update_firmware",
            missing_fields=missing,
        )

    if ("공장" in msg and "초기화" in msg) or "factory reset" in lowered:
        if device_type is None:
            missing.append("device_type")
        return DeviceActionPlan(
            intent="device_control",
            device_id=device_id,
            device_type=device_type,
            action="factory_reset",
            missing_fields=missing,
        )

    manualish = any(
        k in lowered
        for k in [
            "방법",
            "가이드",
            "매뉴얼",
            "알려",
            "how",
        ]
    )
    wifi_app = any(k in lowered for k in ["앱", "wi-fi", "wifi", "와이파이", "연결"])
    if wifi_app and not error_code and ("연결" in msg or "앱" in msg):
        return DeviceActionPlan(
            intent="manual_info",
            device_id=device_id or resolve_device_id("remote_hub"),
            device_type=device_type or "remote_hub",
            question=msg,
            missing_fields=missing,
        )

    trouble_markers = [
        "에러",
        "오류",
        "안 돼",
        "안돼",
        "안됨",
        "약해",
        "원격진단",
        "진단",
        "뜨",
    ]
    if error_code or any(t in msg for t in trouble_markers):
        if device_type is None and device_id is None:
            missing.append("device_type")
        return DeviceActionPlan(
            intent="troubleshooting",
            device_id=device_id,
            device_type=device_type,
            error_code=error_code,
            symptom=msg,
            missing_fields=missing,
        )

    if manualish:
        return DeviceActionPlan(
            intent="manual_info",
            device_id=device_id,
            device_type=device_type,
            question=msg,
            missing_fields=missing,
        )

    return DeviceActionPlan(
        intent="manual_info",
        device_id=device_id,
        device_type=device_type,
        question=msg,
        missing_fields=missing,
    )

## Step 6. End-to-End — `run_support_turn()` (세션·2회 확인·역할 가드·Tool trace)

In [8]:
SESSIONS: dict[str, dict[str, Any]] = {}


def _session_get(session_id: str) -> dict[str, Any]:
    if session_id not in SESSIONS:
        SESSIONS[session_id] = {"pending_dangerous": None}
    return SESSIONS[session_id]


def _next_fw_version(current: str) -> str:
    parts = current.split(".")
    try:
        parts[-1] = str(int(parts[-1]) + 1)
        return ".".join(parts)
    except Exception:
        return current


def run_support_turn(message: str, role: Role = "customer", session_id: str = "demo") -> SupportReport:
    sess = _session_get(session_id)
    trace: list[dict[str, Any]] = []
    pending = sess.get("pending_dangerous")

    if pending and _is_no(message):
        sess["pending_dangerous"] = None
        return SupportReport(
            title="danger_cancelled",
            intent="device_control",
            severity="low",
            reply="위험 작업을 취소했습니다. 다른 도움이 필요하면 말씀해 주세요.",
            tool_trace=trace,
        )

    if pending and not _is_yes(message) and not _is_no(message):
        return SupportReport(
            title="danger_pending_other",
            intent="device_control",
            severity="low",
            reply="진행 중인 위험 작업 확인이 있습니다. 먼저 '예' 또는 아니오로 답해 주세요.",
            needs_confirmation=True,
            pending_action=pending,
            tool_trace=trace,
        )

    if pending and _is_yes(message):
        pending["confirm_round"] = int(pending.get("confirm_round", 0)) + 1
        cr = pending["confirm_round"]
        if cr < 2:
            sess["pending_dangerous"] = pending
            return SupportReport(
                title="danger_confirm_round2",
                intent="device_control",
                severity="medium",
                reply=(
                    "한 번 더 확인합니다. 전원이 안정적이고 업데이트 중 전원을 끄지 않겠다는 점을 이해하셨나요? "
                    "계속하려면 다시 '예'라고 답해 주세요."
                ),
                needs_confirmation=True,
                pending_action=pending,
                tool_trace=trace,
            )
        # 두 번째 예 이후 실행 시도
        if not role_may_use_sensitive_tools(role):
            sess["pending_dangerous"] = None
            return SupportReport(
                title="danger_denied_role",
                intent="device_control",
                severity="high",
                reply=(
                    "고객 권한으로는 펌웨어/초기화 같은 위험 작업을 원격 실행할 수 없습니다. "
                    "지원(support) 또는 관리자(admin) 세션에서 다시 요청해 주세요."
                ),
                tool_trace=trace,
            )
        kind = pending.get("kind")
        dev = pending.get("device_id")
        target = pending.get("target_version")
        if kind == "update_firmware":
            r = update_firmware(dev, target, confirmed=True)
            trace.append({"tool": "update_firmware", "result": r})
        elif kind == "factory_reset":
            r = factory_reset(dev, confirmed=True)
            trace.append({"tool": "factory_reset", "result": r})
        else:
            r = {"ok": False, "error": {"code": "UNKNOWN_PENDING", "message": kind}}
        sess["pending_dangerous"] = None
        ok = r.get("ok")
        return SupportReport(
            title="danger_executed",
            intent="device_control",
            severity="medium" if ok else "high",
            reply=("요청하신 위험 작업을 실행했습니다(시뮬레이션)." if ok else f"실행 실패: {r}"),
            recommended_actions=["작업 후 기기 재시작 및 앱 동기화를 확인하세요."],
            tool_trace=trace,
        )

    plan = build_action_plan(message)

    if plan.intent == "handoff" or plan.safety_risk:
        summary = f"안전 이슈 의심: {plan.symptom or message}"
        ticket_result = None
        if role_may_use_sensitive_tools(role):
            dev = plan.device_id or "UNKNOWN"
            ticket_result = create_service_ticket(dev, summary, "high")
            trace.append({"tool": "create_service_ticket", "result": ticket_result})
        return SupportReport(
            title="safety_handoff",
            intent="handoff",
            severity="high",
            reply=(
                "즉시 사용을 중지하고 전원을 차단하세요. 환기가 가능하면 창문을 열고, 화재·가스 냄새가 지속되면 대피 후 긴급 연락망을 이용하세요. "
                "원격 조작 없이 AS/안전 점검(핸드오프)이 필요합니다."
            ),
            evidence=[policies.get("dangerous_actions", "")],
            recommended_actions=["전원 차단", "가연성 물질과 거리 두기", "공식 서비스 센터 연락"],
            needs_handoff=True,
            ticket=ticket_result.get("ticket") if ticket_result and ticket_result.get("ok") else None,
            tool_trace=trace,
        )

    dtype = plan.device_type
    dev_id = plan.device_id

    if plan.intent == "device_control" and plan.action in ("update_firmware", "factory_reset"):
        if dev_id is None and dtype:
            dev_id = resolve_device_id(dtype)
        if dev_id is None:
            return SupportReport(
                title="danger_missing_device",
                intent="device_control",
                severity="medium",
                reply="기기를 특정할 수 없습니다. 제품군(예: 에어컨)과 기기 ID(DEV-xxxx)를 알려주세요.",
                follow_up_question="어떤 기기에 대해 작업할까요? (모델명 또는 DEV-번호)",
                tool_trace=trace,
            )
        cur_fw = devices.get(dev_id, {}).get("fw", "0.0.0")
        target_version = _next_fw_version(str(cur_fw))
        sess["pending_dangerous"] = {
            "kind": plan.action,
            "device_id": dev_id,
            "target_version": target_version,
            "confirm_round": 0,
        }
        probe = (
            update_firmware(dev_id, target_version, confirmed=False)
            if plan.action == "update_firmware"
            else factory_reset(dev_id, confirmed=False)
        )
        trace.append({"tool": plan.action, "result": probe})
        return SupportReport(
            title="danger_confirm_round1",
            intent="device_control",
            severity="medium",
            reply=(
                "해당 작업은 기기에 영향을 줄 수 있는 위험 작업입니다. 진행하려면 '예'라고 답해 주세요. "
                "(두 번째 '예'에서 최종 실행됩니다.)"
            ),
            needs_confirmation=True,
            pending_action=sess["pending_dangerous"],
            tool_trace=trace,
        )

    evidence: list[str] = []
    rec: list[str] = []
    follow: Optional[str] = None

    if plan.missing_fields and plan.intent == "troubleshooting":
        follow = "기기 모델명, 증상, 에러 코드(있다면)를 알려주시면 더 정확히 안내할 수 있어요."

    hits = search_manual(message, dtype or plan.device_type)
    if hits:
        for h in hits:
            evidence.append(f"{h['citation']} {h['text']}")
    else:
        follow = follow or (
            "매뉴얼에서 직접적인 일치 항목을 찾지 못했습니다(NO_HIT). "
            "기기 모델명, 증상, 에러 코드를 알려주세요."
        )

    if plan.intent in ("manual_info", "troubleshooting") and dev_id:
        st = get_device_status(dev_id)
        trace.append({"tool": "get_device_status", "result": st})

    if plan.intent == "troubleshooting" and dev_id:
        ev = get_recent_events(dev_id, limit=3)
        trace.append({"tool": "get_recent_events", "result": ev})
        if role_may_use_sensitive_tools(role):
            diag = run_remote_diagnosis(dev_id)
            trace.append({"tool": "run_remote_diagnosis", "result": diag})
            if diag.get("ok"):
                rec.extend(diag.get("findings", []))
        else:
            rec.append("원격 진단은 지원/관리 권한에서만 실행됩니다. 아래 매뉴얼 절차를 먼저 따라 주세요.")

    if plan.intent == "troubleshooting" and (dtype == "washing_machine" or plan.error_code == "5C"):
        rec.extend(
            [
                "전원 차단 후 배수 필터를 점검하고 이물을 제거합니다.",
                "배수 호스 꺾임/막힘을 확인합니다.",
            ]
        )
    if plan.intent == "troubleshooting" and dtype == "robot_vacuum":
        rec.extend(["필터 분리·세척 후 완전 건조", "메인 브러시/사이드 브러시 감김 여부 확인"])

    if plan.intent == "manual_info" and dtype == "remote_hub":
        rec.extend(["2.4GHz 대역 연결 확인", "앱에서 기기 재등록", "공유기 재부팅"])

    sev: Literal["low", "medium", "high"] = "low"
    if plan.intent == "troubleshooting":
        sev = "medium"

    reply_parts = []
    if hits:
        reply_parts.append("매뉴얼 근거를 바탕으로 안내드립니다.")
    elif follow:
        reply_parts.append("매뉴얼 직접 일치 항목이 없어 추가 정보가 필요합니다.")
    if rec:
        reply_parts.append("추천 조치: " + " / ".join(rec[:5]))
    reply = " ".join(reply_parts) if reply_parts else "요청을 처리했습니다."

    return SupportReport(
        title="support_turn",
        intent=plan.intent,
        severity=sev,
        reply=reply,
        evidence=evidence,
        recommended_actions=rec,
        follow_up_question=follow,
        tool_trace=trace,
    )

## Step 7. 시나리오 시연 로그 (5개) + 역할(customer) 가드 예시

In [9]:
def print_report(item_id: str, report: SupportReport) -> None:
    print("=" * 80)
    print(item_id)
    print(json.dumps(report.model_dump(), ensure_ascii=False, indent=2))


SESSIONS.clear()
for item in scenarios:
    sid = item["id"]
    report = run_support_turn(item["input"], role="support", session_id=sid)
    print_report(sid, report)
    assert report.intent == item["expected_intent"], (sid, report.intent, item["expected_intent"])

print("\n--- 펌웨어 2턴 확인 + support 실행 ---")
SESSIONS.pop("scenario-4", None)
r1 = run_support_turn("에어컨 펌웨어 업데이트 해줘", role="support", session_id="scenario-4")
print_report("scenario-4 / turn1", r1)
assert r1.needs_confirmation is True
r2 = run_support_turn("예", role="support", session_id="scenario-4")
print_report("scenario-4 / turn2", r2)
assert r2.needs_confirmation is True
r3 = run_support_turn("예", role="support", session_id="scenario-4")
print_report("scenario-4 / turn3", r3)
assert r3.title == "danger_executed"

print("\n--- 동일 요청이나 customer 권한이면 최종 실행 거부 ---")
SESSIONS.pop("fw-customer", None)
run_support_turn("에어컨 펌웨어 업데이트 해줘", role="customer", session_id="fw-customer")
run_support_turn("예", role="customer", session_id="fw-customer")
rdeny = run_support_turn("예", role="customer", session_id="fw-customer")
print_report("fw-customer / denied", rdeny)
assert "고객 권한" in rdeny.reply

print("\n--- NO_HIT 추가 질문 예시(의도적으로 희귀 키워드) ---")
SESSIONS.pop("nohit", None)
rn = run_support_turn("ZZZ_UNKNOWN_TOKEN_12345", role="support", session_id="nohit")
print_report("nohit", rn)
assert rn.follow_up_question is not None

scenario-1
{
  "title": "support_turn",
  "intent": "troubleshooting",
  "severity": "medium",
  "reply": "매뉴얼 근거를 바탕으로 안내드립니다. 추천 조치: 배수 타임아웃(5C) 패턴: 드레인 라인/필터 막힘 가능성이 높습니다. / 전원 차단 후 배수 필터를 점검하고 이물을 제거합니다. / 배수 호스 꺾임/막힘을 확인합니다.",
  "evidence": [
    "[manual:manual_washer:chunk1] [세탁기 트러블슈팅 가이드] 5C(배수) 에러가 보이면 전원을 끄고 배수 필터 이물질을 먼저 제거합니다. 배수 호스가 꺾이거나 막히지 않았는지 확인합니다. 응급 배수 후에도 같은 증상이 반복되면 서비스 점검이 필요합니다."
  ],
  "recommended_actions": [
    "배수 타임아웃(5C) 패턴: 드레인 라인/필터 막힘 가능성이 높습니다.",
    "전원 차단 후 배수 필터를 점검하고 이물을 제거합니다.",
    "배수 호스 꺾임/막힘을 확인합니다."
  ],
  "follow_up_question": null,
  "needs_handoff": false,
  "needs_confirmation": false,
  "pending_action": null,
  "ticket": null,
  "tool_trace": [
    {
      "tool": "get_device_status",
      "result": {
        "ok": true,
        "device_id": "DEV-2002",
        "status": {
          "device_type": "washing_machine",
          "model": "Washer-A",
          "status": "online",
          "mode": "standby",
          "fw": "2.1.7",
    

## 참고

- 스켈레톤: `Project1_SmartHome_Care_Copilot_Skeleton.ipynb`
- 가이드: `GUIDE.md`
- OpenAI structured output / LangGraph 로 동일 흐름을 옮기려면 `DeviceActionPlan` 파싱과 노드 분기만 교체하면 됩니다.